<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 50 GraphQL

The notebook judges the GraphQL API of the service through the binary `bausteine-graphql`, which
boots the service through `aosabi::embed` exactly as the Web Worker of the page boots it, so the
answers are those the page receives: the schema inferred from the plain Rust types, the scenes and
the examples, every mutation with its checks, the tree of the groups, new bodies, the revision, the
export to POV-Ray with the date and the time the page hands in, the limits of the store and the
answer to a body that is no request. A cell that proves an entry of the register
`Documentation/quirks.md` carries the tag its column Test names, and its checks name the ID of the
entry.

In [ ]:
engine_binary = None  # the path of bausteine-graphql, the debug build by default

In [ ]:
import sys, json, math
sys.path.insert(0, '..')
from harness import *

In [ ]:
import re
service = Engine(engine_binary)

## 1. The schema

The service infers its schema from its plain Rust types, so introspection names every type the guide
of the API lists. The surface of a texture is a plain enum of Rust, which the schema names as the
enum `Surface` with the families of the ray tracer in their order.

In [ ]:
schema = service.data('{ __schema { types { name } queryType { fields { name } } mutationType { fields { name } } subscriptionType { fields { name } } } }')['__schema']
types = {t['name'] for t in schema['types']}
for name in ['Scene', 'Item', 'Kind', 'Vector', 'KindInfo', 'Texture', 'Material', 'Surface', 'Example', 'AddItemInput', 'UpdateItemInput']:
    check(name in types, f'the schema holds the type {name}')
SURFACES = ['PLAIN', 'METAL', 'GLASS', 'WOOD', 'MARBLE', 'GRANITE', 'STRIPES']
surfaces = [v['name'] for v in service.data('{ __type(name: "Surface") { enumValues { name } } }')['__type']['enumValues']]
check(surfaces == SURFACES, 'the enum Surface names the seven families of the ray tracer in their order', surfaces)
fields = lambda kind: sorted(f['name'] for f in schema[kind]['fields'])
check(fields('queryType') == ['examples', 'kinds', 'scene', 'scenes', 'textures'], 'the queries', fields('queryType'))
check(fields('mutationType') == ['addItem', 'createScene', 'deleteScene', 'moveItem', 'removeItem', 'renameScene', 'replaceItems', 'updateItem'], 'the mutations', fields('mutationType'))
check(schema['subscriptionType'] is None, 'the API has no subscription, the page owns its one worker')

## 2. The lists

The kinds carry the type codes of version 1.3 and the names its interface gives them, `Disc` for the
disc as the body bar, the menu Objekte and the list of bodies CB1 call it, so the API, the examples
and the page name every body alike. The declaration and the instance keep readable labels. The
textures are the 43 names that `Fill_CB3_Texi` of `POV.bas` puts into the list of F6, in its order
and with `Lightening1`, the name POV-Ray 3.1 knows, which the notebook reads from the file at the
root. Each texture answers the material the ray tracer renders it with, its family, its two colours
as `#rrggbb` and its reflection, the numbers of `bausteine_core::texture::TEXTURES` that the devtool
lists, so the swatch of the tab Oberfläche paints `Jade` green and `Gold_Metal` gold.

In [ ]:
d = service.data('{ kinds { kind label code group } textures { name group } examples { key title } }')
check([k['code'] for k in d['kinds']] == [1, 2, 3, 4, 5, 6, 7, 8, 101, 102, 103, 104, 105, 200], 'the kinds carry the type codes of version 1.3')
check([k['label'] for k in d['kinds'][8:]] == ['Vereinigung', 'Differenz', 'Durchschnitt', 'Summe', 'Neu', 'Objekt'], 'R-29 the groups, the declaration and the instance carry their German names')
check([k['label'] for k in d['kinds'][:8]] == ['Kugel', 'Kegel', 'Zylinder', 'Würfel', 'Quader', 'Pyramide', 'Prisma', 'Disc'], 'R-29 the bodies carry the names of version 1.3, Disc for the disc', [k['label'] for k in d['kinds'][:8]])
fill = (ROOT / 'POV.bas').read_bytes().decode('latin-1')
cb3 = re.findall(r'F6\.CB3\.AddItem "([^"]+)"', fill[fill.index('Sub Fill_CB3_Texi'):])
names = [t['name'] for t in d['textures']]
check(len(names) == 43 and names == cb3, 'M-45 the 43 textures of Fill_CB3_Texi of version 1.3 in their order', names, cb3)
check('Lightening1' in names and 'Lightning1' not in names, 'M-44 the list keeps Lightening1, the name of POV-Ray 3.1')
table = devtool('textures')
served = service.data('{ textures { name material { surface base accent reflection } } }')['textures']
hexed = lambda rgb: '#' + ''.join(f'{math.floor(c * 255 + 0.5):02x}' for c in rgb)
wrong = [(t['name'], s['material']) for t, s in zip(table, served)
         if s['material'] != {'surface': SURFACES[t['surface']], 'base': hexed(t['base']), 'accent': hexed(t['accent']), 'reflection': t['reflection']}]
check([s['name'] for s in served] == [t['name'] for t in table] and wrong == [],
      'CRIT-F4-10 every texture answers the material of the ray tracer, its colours as #rrggbb', wrong)
material = {s['name']: s['material'] for s in served}
check(material['Jade'] == {'surface': 'MARBLE', 'base': '#2e8c52', 'accent': '#8cd99e', 'reflection': 0.2}
      and material['Gold_Metal'] == {'surface': 'METAL', 'base': '#f2bf40', 'accent': '#000000', 'reflection': 0.65},
      'P-151a Jade is a green marble and Gold_Metal a gold metal', material['Jade'], material['Gold_Metal'])
check([e['key'] for e in d['examples']] == ['bausteine', 'haus', 'werkstueck', 'kugeln', 'roulette'], 'the five examples')
blocks = service.data('mutation { createScene(example: "bausteine") { id items { name kind } } }')['createScene']
check([(i['name'], i['kind']) for i in blocks['items'] if i['kind'] == 'DISC'] == [('Disc', 'DISC')], 'the disc of the example bausteine is named Disc', blocks['items'])
service.data('mutation($s: ID!) { deleteScene(id: $s) }', {'s': blocks['id']})

## 3. Creating scenes

In [ ]:
ITEM = 'id name kind parent position { x y z } size { x y z } rotation { x y z } color texture innerRadius'
empty = service.data('mutation { createScene { id name revision items { id } } }')['createScene']
check(empty['name'] == 'Neue Szene' and empty['items'] == [] and empty['revision'] == 0, 'an empty scene', empty)
house = service.data(f'mutation {{ createScene(example: "haus", name: "  Mein Haus  ") {{ id name items {{ {ITEM} }} }} }}')['createScene']
check(house['name'] == 'Mein Haus' and len(house['items']) == 6, 'a scene from an example keeps its items and a trimmed name')
walls = house['items'][0]
check(walls['kind'] == 'DIFFERENCE' and [i['parent'] for i in house['items'][1:4]] == [walls['id']] * 3, 'the walls are a difference of three cuboids')
bad = service.graphql('mutation { createScene(example: "nope") { id } }')
check(bad['errors'][0]['extensions']['code'] == 'BAD_USER_INPUT', 'an unknown example is refused', bad)
sid = empty['id']

## 4. Adding and checking items

An item takes the defaults of its kind and no texture, `null`, which the combo shows as `Keine`. A
colour is a value `#rrggbb`, black among them, a texture is stored by its name, a name holds up to
60 characters without padding, and a number is finite and at most 10000 in size. The service refuses
every other value and names the field.

In [ ]:
added = service.data(f'mutation($s: ID!) {{ addItem(scene: $s, input: {{ kind: SPHERE }}) {{ {ITEM} }} }}', {'s': sid})['addItem']
check(added['name'] == 'Kugel' and added['color'] == '#d9d9d9' and added['size'] == {'x': 1, 'y': 1, 'z': 1}, 'an item takes the defaults', added)
check(added['texture'] is None, 'R-59 an item without a texture stores null, which the combo shows as Keine', added)
refusals = {
    'color': '{ kind: CUBE, color: "red" }',
    'texture': '{ kind: CUBE, texture: "Plastic" }',
    'position': '{ kind: CUBE, position: { x: 1e9, y: 0, z: 0 } }',
    'innerRadius': '{ kind: DISC, innerRadius: -1 }',
    'parent': f'{{ kind: CUBE, parent: "{added["id"]}" }}',
}
proves = {'color': 'Q-36 ', 'texture': 'M-46 ', 'position': 'F4-Q63 ', 'innerRadius': 'F4-Q63 ', 'parent': ''}
for field, literal in refusals.items():
    body = service.graphql(f'mutation($s: ID!) {{ addItem(scene: $s, input: {literal}) {{ id }} }}', {'s': sid})
    ext = body.get('errors', [{}])[0].get('extensions', {})
    check(ext.get('code') == 'BAD_USER_INPUT' and ext.get('details', {}).get('field') == field, f'{proves[field]}a wrong {field} is refused', body)
missing = service.graphql('mutation { addItem(scene: "none", input: { kind: CUBE }) { id } }')
check(missing['errors'][0]['extensions']['code'] == 'SCENE_NOT_FOUND', 'a scene that does not exist is named so', missing)

probe = service.data('mutation { createScene(name: "Probe") { id } }')['createScene']['id']
def accepted(literal):
    return service.data(f'mutation($s: ID!) {{ addItem(scene: $s, input: {literal}) {{ {ITEM} }} }}', {'s': probe})['addItem']
check(accepted('{ kind: CUBE, color: "#000000" }')['color'] == '#000000', 'Q-36 black is a colour like any other, #000000')
check(accepted('{ kind: CUBE, texture: "Jade" }')['texture'] == 'Jade', 'M-46 the API stores the name of the texture, Jade')
cut = accepted('{ kind: CUBE, name: "%s" }' % ('Werkstück' * 7))['name']
padded = accepted('{ kind: CUBE, name: "  CSG-Durchschnitt  " }')['name']
check(cut == ('Werkstück' * 7)[:60] and padded == 'CSG-Durchschnitt', 'F4-Q57 a name holds up to 60 characters and carries no padding blanks', cut, padded)
check(accepted('{ kind: CUBE, position: { x: 10000, y: -10000, z: 0 } }')['position'] == {'x': 10000, 'y': -10000, 'z': 0},
      'F4-Q63 the service takes numbers up to 10000 in size')
far = service.graphql('mutation($s: ID!) { addItem(scene: $s, input: { kind: CUBE, size: { x: 10000.5, y: 1, z: 1 } }) { id } }', {'s': probe})
check(far['errors'][0]['extensions']['details']['field'] == 'size', 'F4-Q63 a number greater than 10000 is refused with its field', far)
service.data('mutation($s: ID!) { deleteScene(id: $s) }', {'s': probe})

## 5. Groups, moves and removals

An item moves into a group with `parent`, to the top with an empty `parent`, and never into a group
below itself. Removing a group removes everything in it.

In [ ]:
def scene(sid):
    return service.data(f'query($s: ID!) {{ scene(id: $s) {{ revision items {{ {ITEM} }} }} }}', {'s': sid})['scene']
g = service.data('mutation($s: ID!) { addItem(scene: $s, input: { kind: UNION, name: "Gruppe" }) { id } }', {'s': sid})['addItem']['id']
h = service.data('mutation($s: ID!, $p: ID) { addItem(scene: $s, input: { kind: DIFFERENCE, parent: $p }) { id } }', {'s': sid, 'p': g})['addItem']['id']
service.data('mutation($s: ID!, $i: ID!, $p: ID) { updateItem(scene: $s, id: $i, input: { parent: $p }) { id } }', {'s': sid, 'i': added['id'], 'p': h})
items = scene(sid)['items']
check([i['id'] for i in items] == [g, h, added['id']], 'every group stands before its children', [i['name'] for i in items])
loop = service.graphql('mutation($s: ID!, $i: ID!, $p: ID) { updateItem(scene: $s, id: $i, input: { parent: $p }) { id } }', {'s': sid, 'i': g, 'p': h})
check(loop['errors'][0]['extensions']['details']['field'] == 'parent', 'a group never moves into a group below itself', loop)
top = service.data('mutation($s: ID!, $i: ID!) { updateItem(scene: $s, id: $i, input: { parent: "" }) { parent } }', {'s': sid, 'i': added['id']})['updateItem']
check(top['parent'] is None, 'an empty parent moves an item to the top')
service.data('mutation($s: ID!, $i: ID!) { removeItem(scene: $s, id: $i) { id } }', {'s': sid, 'i': g})
check([i['id'] for i in scene(sid)['items']] == [added['id']], 'F4-Q53 removing a group removes everything in it, so the list stays balanced')

## 6. New bodies

A declaration defines a new body, `#Neu` of version 1.3, and an instance places it again. An
instance names a declaration and never lies inside it.

In [ ]:
dec = service.data('mutation($s: ID!) { addItem(scene: $s, input: { kind: DECLARE, name: "Werkstück1" }) { id } }', {'s': sid})['addItem']['id']
service.data('mutation($s: ID!, $p: ID) { addItem(scene: $s, input: { kind: CUBE, parent: $p }) { id } }', {'s': sid, 'p': dec})
inst = service.data('mutation($s: ID!, $src: ID) { addItem(scene: $s, input: { kind: INSTANCE, source: $src, position: { x: 5, y: 0, z: 0 } }) { id source kind } }', {'s': sid, 'src': dec})['addItem']
check(inst['source'] == dec and inst['kind'] == 'INSTANCE', 'an instance places the declaration')
inside = service.graphql('mutation($s: ID!, $src: ID, $p: ID) { addItem(scene: $s, input: { kind: INSTANCE, source: $src, parent: $p }) { id } }', {'s': sid, 'src': dec, 'p': dec})
check(inside['errors'][0]['extensions']['details']['field'] == 'source', 'an instance never lies inside its declaration', inside)
nosrc = service.graphql('mutation($s: ID!) { addItem(scene: $s, input: { kind: INSTANCE }) { id } }', {'s': sid})
check(nosrc['errors'][0]['extensions']['details']['field'] == 'source', 'an instance needs a declaration', nosrc)
moved = service.graphql('mutation($s: ID!, $i: ID!, $p: ID) { updateItem(scene: $s, id: $i, input: { parent: $p }) { id } }', {'s': sid, 'i': inst['id'], 'p': dec})
check(moved['errors'][0]['extensions']['details']['field'] == 'parent', 'an instance never moves into its declaration', moved)
pov = service.data('query($s: ID!) { scene(id: $s) { pov } }', {'s': sid})['scene']['pov']
check('#declare Neu_Werkstueck1 = union {' in pov and pov.count('object { Neu_Werkstueck1') == 2, 'the export declares the body once and places it twice')
for gone in (inst['id'], dec):
    service.data('mutation($s: ID!, $i: ID!) { removeItem(scene: $s, id: $i) { id } }', {'s': sid, 'i': gone})

## 7. Replacing the items keeps their ids

The page undoes a change by replacing the items with those before it, so the ids an undo hands back
survive.

In [ ]:
before = scene(sid)
inputs = [{k: v for k, v in i.items()} for i in house['items']]
replaced = service.data(f'mutation($s: ID!, $items: [AddItemInput!]!) {{ replaceItems(scene: $s, items: $items) {{ revision items {{ id parent }} }} }}', {'s': sid, 'items': inputs})['replaceItems']
check([i['id'] for i in replaced['items']] == [i['id'] for i in house['items']], 'the ids survive a replacement')
check(replaced['revision'] == before['revision'] + 1, 'a replacement counts one revision', replaced['revision'])

## 8. The list OBJEKTE

Version 1.3 writes one line per object into the list OBJEKTE, `CB2_ADD` of F4.frm: the number, two
blanks per level of the CSG, a blank, the marks of the open groups and the name. An opener of the
CSG stands at its level, its members carry its mark, `°` for a union, `-` for a difference, `^` for
an intersection and `+` for a merge, and `CSG-Ende` stands back at the level of the opener. `#Neu`
takes no mark and no level, and a declaration ends with a line of its name. The field `list` writes
the same lines for the tree of 2.0, and the thesis shows the example on p. 95: `1 #Neu Werkstück1`,
`2 CSG-Differenz`, `3   -Quader`, `4   -Zylinder`, `5 CSG-Ende`, `6 Werkstück1`.

In [ ]:
thesis = service.data('mutation { createScene(name: "Seite 95") { id } }')['createScene']['id']
add = lambda input, after=None: service.data('mutation($s: ID!, $i: AddItemInput!, $a: ID) { addItem(scene: $s, input: $i, after: $a) { id } }', {'s': thesis, 'i': input, 'a': after})['addItem']['id']
neu = add({'kind': 'DECLARE', 'name': 'Werkstück1'})
diff = add({'kind': 'DIFFERENCE', 'name': 'CSG-Differenz', 'parent': neu})
add({'kind': 'CUBOID', 'name': 'Quader', 'parent': diff})
add({'kind': 'CYLINDER', 'name': 'Zylinder', 'parent': diff})
lines = service.data('query($s: ID!) { scene(id: $s) { list { text item end } } }', {'s': thesis})['scene']['list']
texts = [l['text'] for l in lines]
check(texts == ['0 Raster', '1 #Neu Werkstück1', '2 CSG-Differenz', '3   -Quader', '4   -Zylinder', '5 CSG-Ende', '6 Werkstück1'], 'T-p95c the list of p. 95 of the thesis, with 0 Raster first', texts)
check(lines[0]['item'] is None and lines[5] == {'text': '5 CSG-Ende', 'item': diff, 'end': True} and lines[6]['item'] == neu and lines[6]['end'], 'each line names its item, an end line its group')
made = service.data('mutation { createScene(example: "werkstueck") { id items { id } } }')['createScene']
nested = made['id']
u = service.data('mutation($s: ID!, $p: ID) { addItem(scene: $s, input: { kind: UNION, name: "CSG-Vereinigung", parent: $p }) { id } }', {'s': nested, 'p': made['items'][0]['id']})['addItem']['id']
service.data('mutation($s: ID!, $p: ID) { addItem(scene: $s, input: { kind: SPHERE, name: "Kugel", parent: $p }) { id } }', {'s': nested, 'p': u})
texts = [l['text'] for l in service.data('query($s: ID!) { scene(id: $s) { list { text } } }', {'s': nested})['scene']['list']]
check(texts == ['0 Raster', '1 Werkstück', '2   -Rohling', '3   -Bohrung', '4   -CSG-Vereinigung', '5     -°Kugel', '6   -CSG-Ende', '7 CSG-Ende'], 'T-p48b the marks add up with the levels', texts)

## 9. Inserting behind the selected line and moving

The button Einfügen of 1.3 inserts behind the selected line. `addItem` takes the item the new one
follows with `after`: a member of the same group, or the group itself, which the new item then
opens. `moveItem` puts an item behind another member of its group, or at its front, since the order
of the members is the meaning of a difference.

In [ ]:
first = add({'kind': 'CUBE', 'name': 'Erster', 'parent': diff}, diff)
names = lambda: [l['text'].split(' ', 1)[1].strip() for l in service.data('query($s: ID!) { scene(id: $s) { list { text } } }', {'s': thesis})['scene']['list']]
check(names()[2:6] == ['CSG-Differenz', '-Erster', '-Quader', '-Zylinder'], 'behind its group a new item opens it', names())
quader = [l['item'] for l in service.data('query($s: ID!) { scene(id: $s) { list { text item } } }', {'s': thesis})['scene']['list'] if l['text'].endswith('-Quader')][0]
add({'kind': 'PRISM', 'name': 'Prisma', 'parent': diff}, quader)
check(names()[3:7] == ['-Erster', '-Quader', '-Prisma', '-Zylinder'], 'F4-Q39 Einfügen behind a member puts the new item right after it, not at the end', names())
service.data('mutation($s: ID!, $i: ID!, $a: ID) { moveItem(scene: $s, id: $i, after: $a) { id } }', {'s': thesis, 'i': first, 'a': None})
moved = service.data('mutation($s: ID!, $i: ID!, $a: ID) { moveItem(scene: $s, id: $i, after: $a) { id } }', {'s': thesis, 'i': first, 'a': quader})
check(names()[3:7] == ['-Quader', '-Erster', '-Prisma', '-Zylinder'], 'moveItem reorders the members of a group', names())
wrong = service.graphql('mutation($s: ID!, $i: ID!, $a: ID) { moveItem(scene: $s, id: $i, after: $a) { id } }', {'s': thesis, 'i': first, 'a': neu})
check(wrong['errors'][0]['extensions']['details']['field'] == 'after', 'an item moves only within its group', wrong)
kind = service.data('mutation($s: ID!, $i: ID!) { updateItem(scene: $s, id: $i, input: { kind: CYLINDER }) { kind } }', {'s': thesis, 'i': quader})['updateItem']['kind']
check(kind == 'CYLINDER', 'Ändern turns a body into another body, as 1.3 runs the procedure of the body of CB1', kind)
regroup = service.data('mutation($s: ID!, $i: ID!) { updateItem(scene: $s, id: $i, input: { kind: UNION }) { kind } }', {'s': thesis, 'i': diff})['updateItem']['kind']
check(regroup == 'UNION', 'a group of the CSG becomes another group', regroup)
for target, kind in [(diff, 'SPHERE'), (quader, 'MERGE'), (neu, 'UNION')]:
    refused = service.graphql('mutation($s: ID!, $i: ID!, $k: Kind) { updateItem(scene: $s, id: $i, input: { kind: $k }) { kind } }', {'s': thesis, 'i': target, 'k': kind})
    check(refused['errors'][0]['extensions']['details']['field'] == 'kind', f'a {kind} is refused across the families', refused)

## 10. The head of the export

`Def_Pov` of 1.3 writes a head with the version, the date and the time, each line of comment
followed by an empty line. The field `pov` writes that layout with `classic` and the date and time
the page hands in, since the module of WebAssembly reads no clock, and the compact head of 2.0
without. The date is `TT.MM.JJJJ` and the time `HH:MM:SS`, the German `Date` and `Time` of Visual
Basic, as `ausgabe.pov` of 1.3 shows them: `06.03.2002` and `21:03:31`. The service writes both as
they come, a line break as a blank, and without them the classic head leaves their places empty and
the head of 2.0 names no date.

In [ ]:
head = service.data('query($s: ID!) { scene(id: $s) { pov(classic: true, date: "06.11.2026", time: "16:20:00") } }', {'s': thesis})['scene']['pov']
check(head.startswith('//==========================================================\n\n// 3D-Bausteine Version 2.0 (klassisch) nach 3D-Baukasten 1.3 (c) 2002 H.Kleber\n\n// POV 2 Datei Datum: 06.11.2026 Zeit: 16:20:00\n\n'), 'the classic head has the layout of Def_Pov', head[:200])
modern = service.data('query($s: ID!) { scene(id: $s) { pov(date: "06.11.2026", time: "16:20:00") } }', {'s': thesis})['scene']['pov']
check(modern.splitlines()[1:3] == ['// 3D-Bausteine 2.0 nach 3D-Baukasten 1.3 (H. Kleber 2002)', '// Szene für POV-Ray 3, Datum: 06.11.2026 Zeit: 16:20:00'], 'the head of 2.0', modern.splitlines()[:4])
check(head.split('#include', 1)[1] == modern.split('#include', 1)[1], 'both heads share everything after the comment')
pov13 = lambda args: service.data(f'query($s: ID!) {{ scene(id: $s) {{ pov({args}) }} }}', {'s': thesis})['scene']['pov']
dated = pov13('classic: true, date: "06.03.2002", time: "21:03:31"').splitlines()[4]
golden = (ROOT / 'ausgabe.pov').read_text(encoding='cp1252').splitlines()[4]
check(dated == golden and re.fullmatch(r'// POV 2 Datei Datum: \d\d\.\d\d\.\d{4} Zeit: \d\d:\d\d:\d\d', dated),
      'R-56 the date TT.MM.JJJJ and the time HH:MM:SS the page hands in stand as in the head of ausgabe.pov of 1.3', dated, golden)
broken = pov13('classic: true, date: "06.03.2002\\n// x", time: "21:03:31"').splitlines()
check(broken[4] == '// POV 2 Datei Datum: 06.03.2002 // x Zeit: 21:03:31' and broken[6].startswith('//='), 'a line break in the date becomes a blank, so the head keeps its lines', broken[:8])
check(pov13('classic: true').splitlines()[4] == '// POV 2 Datei Datum:  Zeit: ', 'without a date and a time the classic head leaves their places empty')
check(service.data('query($s: ID!) { scene(id: $s) { pov } }', {'s': thesis})['scene']['pov'].splitlines()[2] == '// Szene für POV-Ray 3', 'without them the head of 2.0 names no date')

## 11. The export and the limits

The field `pov` exports a scene from any camera. A scene holds at most 512 items, and the service
keeps at most 256 scenes in the memory of the worker, the oldest going first, and lists them newest
first. The service builds the text of the export in Rust, so a scene of 512 items exports whole, far
beyond the 64 KB that the text box of Windows 9x of version 1.3 holds.

In [ ]:
pov = service.data('query($s: ID!) { scene(id: $s) { pov(eye: { x: 10, y: 10, z: 5 }) } }', {'s': sid})['scene']['pov']
check('camera { location <10, 5, -10>' in pov and 'difference {' in pov, 'the field pov exports the scene from any camera')
many = [{'kind': 'SPHERE'} for _ in range(513)]
full = service.graphql('mutation($s: ID!, $items: [AddItemInput!]!) { replaceItems(scene: $s, items: $items) { id } }', {'s': sid, 'items': many})
check(full['errors'][0]['extensions']['details']['field'] == 'items', 'T-p52c a scene holds at most 512 items', full)
big = service.data('mutation { createScene(name: "Groß") { id } }')['createScene']['id']
most = [{'kind': 'CUBOID', 'name': f'Quader {i} der großen Szene', 'position': {'x': i * 0.1234, 'y': i, 'z': 2.5}, 'size': {'x': 1.5, 'y': 2.25, 'z': 3.125}} for i in range(512)]
service.data('mutation($s: ID!, $items: [AddItemInput!]!) { replaceItems(scene: $s, items: $items) { id } }', {'s': big, 'items': most})
whole = service.data('query($s: ID!) { scene(id: $s) { pov } }', {'s': big})['scene']['pov']
check(len(whole.encode('utf-8')) > 65536 and '// Objekt 512: Quader 511 der großen Szene' in whole and whole.rstrip().endswith('}'),
      'M-25 the service exports a scene of 512 items whole, beyond 64 KB', len(whole))
service.data('mutation($s: ID!) { deleteScene(id: $s) }', {'s': big})
gone = service.data('mutation($s: ID!) { deleteScene(id: $s) }', {'s': sid})['deleteScene']
check(gone and service.data('query($s: ID!) { scene(id: $s) { id } }', {'s': sid})['scene'] is None, 'a deleted scene is gone')
bodies = [service.graphql('mutation($n: String) { createScene(name: $n) { id } }', {'n': f'Szene {i}'}) for i in range(257)]
check(all('errors' not in b for b in bodies), 'the service creates 257 scenes one after the other', [b for b in bodies if 'errors' in b][:1])
made = [b['data']['createScene']['id'] for b in bodies]
kept = [s['id'] for s in service.data('{ scenes { id } }')['scenes']]
check(len(kept) == 256, 'T-p52c the service keeps 256 scenes', len(kept))
check(kept == made[::-1][:256], 'the oldest scene goes first, and the list names the newest first', kept[:3], made[-3:])
check(service.data('query($s: ID!) { scene(id: $s) { id } }', {'s': made[0]})['scene'] is None, 'the scene created first is gone')

## 12. A body that is no request

The engine answers a body it cannot read with one error and no data, as a GraphQL server over HTTP
answers it, and never ends.

In [ ]:
for line, fault in [('not json', 'is not JSON'), ('[1, 2]', 'is not a JSON object'), ('{"variables": {}}', 'no string field query')]:
    body = service.raw(line)
    check(fault in body['errors'][0]['message'] and 'data' not in body, f'the body {line!r} answers one error', body)
check(service.data('{ textures { name } }')['textures'][0]['name'] == 'Chrome_Metal', 'the engine answers on after them')
schema_text = service.schema()
check('type Query {' in schema_text and 'enum Kind {' in schema_text, 'the engine prints the schema it answers with')

In [ ]:
service.stop()